## **Name: Abdelrhman Mohamed Abdelhady Hodib ID: 2022513643**

## **Assignment3 (Emotions Classification)**

### Importing libraries

In [5]:
# Necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import string
import nltk
nltk.download('wordnet')
nltk.download('stopwords')

/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/c62e74cb-9914-4a18-99a8-08cf34d7bc01/nltk_data..
[nltk_data]     .
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/c62e74cb-9914-4a18-99a8-08cf34d7bc01/nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!


True

### Data Preprocessing

In [7]:
# Load data
data = pd.read_csv('Emotion_classify_Data.csv')

In [8]:
# Data preprocessing
'''
This function performs several operations:

Converts all text to lowercase.
Removes punctuation from the text.
Splits the text into individual words or tokens.
Lemmatizes each word (reduces it to its base or dictionary form) using WordNet.
Removes English stopwords (e.g., 'and', 'the', 'is').

'''

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

data['Preprocessed'] = data['Comment'].apply(preprocess_text)

In [9]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(data['Preprocessed'])
y = data['Emotion']

In [10]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Logistic regression and Naive Bayes

In [12]:
'''
Trains a logistic regression model with X_train and y_train.
Uses this trained model to predict labels for the test dataset X_test.
Calculates the accuracy of these predictions, storing the result in logreg_accuracy.
'''
# Logistic Regression
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)
logreg_accuracy = accuracy_score(y_test, logreg_pred)
logreg_accuracy

0.930976430976431

In [13]:
'''
Naive Bayes (Multinomial):
Trains a Multinomial Naive Bayes model (nb) with X_train and y_train.
Predicts labels for the test dataset X_test using this trained model.
Calculates the accuracy of these predictions, storing the result in nb_accuracy.

Both models are evaluated on the same test dataset to compare how accurately they can predict emotions based on the preprocessed and vectorized textual data.
The accuracy scores obtained for each model can help determine which algorithm performs better for this specific task.
'''
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_accuracy

0.9082491582491582

### Prediction of sample texts

In [15]:
'''
This function encapsulates the entire process of preprocessing the input text,
converting it into a format understandable by the Naive Bayes model,
making a prediction, and returning the predicted emotion category.
'''
# Final function for prediction
def predict_emotion(text):
    preprocessed_text = preprocess_text(text)
    text_vectorized = tfidf.transform([preprocessed_text])
    logreg_prediction = logreg.predict(text_vectorized)
    nb_prediction = nb.predict(text_vectorized)
    return logreg_prediction[0], nb_prediction[0]


In [16]:
# Sample texts for prediction
sample_texts = ["The weather is wonderful today, I'm feeling cheerful.",
"I had a rough day at work, feeling exhausted and stressed.",
"Spending time with loved ones always makes me happy, no matter what."]
# Predicting emotions for sample texts
for text in sample_texts:
    logreg_result, nb_result = predict_emotion(text)
    print(f"Sentance: {text}")
    print(f"Logistic Regression Prediction: {logreg_result}")
    print(f"Naive Bayes Prediction: {nb_result}")
    print("----------------------")

Sentance: The weather is wonderful today, I'm feeling cheerful.
Logistic Regression Prediction: joy
Naive Bayes Prediction: joy
----------------------
Sentance: I had a rough day at work, feeling exhausted and stressed.
Logistic Regression Prediction: anger
Naive Bayes Prediction: anger
----------------------
Sentance: Spending time with loved ones always makes me happy, no matter what.
Logistic Regression Prediction: joy
Naive Bayes Prediction: joy
----------------------
